# Interface between 2D materials: Boron Nitride and Graphene

## Introduction

This notebook creates graphene on four layers of hexagonal boron nitride (h-BN) in a common hexagonal cell, for three registries of graphene on the top h-BN layer and a list of graphene–h-BN distances.

Following the manuscript:
> **Gianluca Giovannetti, Petr A. Khomyakov, Geert Brocks, Paul J. Kelly, and Jeroen van den Brink**
> **Substrate-induced band gap in graphene on hexagonal boron nitride: Ab initio density functional calculations**
> Physical Review B 76, 073103 (2007)
> [DOI: 10.1103/PhysRevB.76.073103](https://doi.org/10.1103/PhysRevB.76.073103)

## 1. Prepare the Environment
### 1.1. Set up the notebook 

Set the following flags to control the notebook behavior 

In [ ]:
LATTICE_CONSTANT = 2.445  # Å, graphene LDA (Giovannetti et al. 2007); h-BN is compressed to it
H_BN_LAYERS = 4
H_BN_INTERLAYER_DISTANCE = 3.24  # Å, the paper's LDA value
VACUUM = 15.0  # Å above graphene

# Registry of graphene on the top h-BN layer, Giovannetti et al. 2007 Fig. 1: one C over B and the other over
# N (a), over N and a hexagon centre (b), over B and a hexagon centre (c). All three run in the paper;
# uncomment to build the others.
STACKINGS = [
    "c",
    # "a",
    # "b",
]
STACKING_SHIFTS = {"a": 0, "b": 1, "c": -1}  # in units of a/√3 along y
# Graphene–h-BN distance, Å. The paper scans 2.5–3.9; three points around (c)'s minimum are active, uncomment
# the rest for the full set.
DISTANCES = [
    3.1, 3.2, 3.3,
    # 2.5, 2.6, 2.7, 2.8, 2.9, 3.0, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9,
]

### 1.2. Install Packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples")

### 1.3. Get input materials and assign `substrate` and `film`
Bulk h-BN (substrate) and graphene (film) are loaded from Standata.

In [ ]:
from mat3ra.standata.materials import Materials
from mat3ra.made.material import Material

film = Material.create(Materials.get_by_name_first_match("C, Graphene, HEX (P6/mmm) 2D (Monolayer), 2dm-3993"))
substrate = Material.create(Materials.get_by_name_first_match("BN, Boron Nitride, HEX (P6_3/mmc) 3D (Bulk), mp-7991"))

### 1.4. Preview Substrate and Film

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

visualize([substrate, film], repetitions=[3, 3, 1], rotation="0x")

## 2. Prepare the slabs

### 2.1. Strain the materials to the common lattice constant
Both materials are strained in-plane to `LATTICE_CONSTANT`, and h-BN along c to `H_BN_INTERLAYER_DISTANCE` between layers.

In [ ]:
from mat3ra.made.tools.build_components.operations.core.modifications.strain.helpers import create_strain

film_scale = LATTICE_CONSTANT / film.lattice.a
substrate_scale = LATTICE_CONSTANT / substrate.lattice.a
substrate_c_scale = 2 * H_BN_INTERLAYER_DISTANCE / substrate.lattice.c
film = create_strain(film, strain_matrix=[[film_scale, 0, 0], [0, film_scale, 0], [0, 0, 1]])
substrate = create_strain(
    substrate, strain_matrix=[[substrate_scale, 0, 0], [0, substrate_scale, 0], [0, 0, substrate_c_scale]]
)

### 2.2. Set the AA' stacking of h-BN
The atoms of the upper layer of the bulk h-BN cell are moved by (1/3, 2/3, 0) in crystal coordinates, so that B sits over N in adjacent layers.

In [ ]:
from mat3ra.made.tools.analyze.other import get_atom_indices_with_condition_on_coordinates
from mat3ra.made.tools.operations.core.unary import translate_atoms

upper_layer_ids = get_atom_indices_with_condition_on_coordinates(substrate, lambda coordinate: coordinate[2] > 0.5)
substrate = translate_atoms(
    substrate, atom_ids=upper_layer_ids, vector=[1 / 3, 2 / 3, 0], use_cartesian_coordinates=False
)

### 2.3. Create Substrate and Film Slabs
The substrate slab has `H_BN_LAYERS` layers of h-BN, the film slab one layer of graphene.

In [ ]:
from mat3ra.made.tools.build.pristine_structures.two_dimensional.slab import SlabConfiguration, SlabBuilder

substrate_slab_config = SlabConfiguration.from_parameters(
    material_or_dict=substrate,
    miller_indices=(0, 0, 1),
    number_of_layers=H_BN_LAYERS // 2,  # two BN layers per bulk cell
    vacuum=0.0,
)

film_slab_config = SlabConfiguration.from_parameters(
    material_or_dict=film,
    miller_indices=(0, 0, 1),
    number_of_layers=1,
    vacuum=0.0,
)

substrate_slab = SlabBuilder().get_material(substrate_slab_config)
film_slab = SlabBuilder().get_material(film_slab_config)

## 3. Create the interfaces

### 3.1. Place graphene at each distance and shift it into each registry
The film is placed on the substrate slab at each of `DISTANCES` and shifted along y by `STACKING_SHIFTS` for each of `STACKINGS`; the registry of each carbon atom is measured on the result.

In [ ]:
import numpy as np
from mat3ra.made.tools.analyze.other import get_average_interlayer_distance
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum
from mat3ra.made.tools.helpers import create_interface_zsl_between_slabs as create_zsl_interface_between_slabs
from mat3ra.made.tools.modify import interface_displace_part


def get_registry(interface):
    elements = np.array(interface.basis.elements.values)
    coordinates = np.array(interface.basis.coordinates.values)
    top_layer = (elements != "C") & np.isclose(coordinates[:, 2], coordinates[elements != "C", 2].max())
    registry = []
    for carbon in coordinates[elements == "C"]:
        in_plane_offsets = (coordinates[top_layer, :2] - carbon[:2] + 0.5) % 1 - 0.5
        atoms_below = elements[top_layer][np.all(np.abs(in_plane_offsets) < 1e-3, axis=1)]
        registry.append(atoms_below[0] if len(atoms_below) else "hollow")
    return registry


interfaces = []
for stacking in STACKINGS:
    for distance in DISTANCES:
        # the builder adds the gap to the vacuum above the film
        interface = create_zsl_interface_between_slabs(
            substrate_slab=substrate_slab, film_slab=film_slab, gap=distance, vacuum=VACUUM - distance
        )
        interface = interface_displace_part(
            interface=interface,
            displacement=[0, STACKING_SHIFTS[stacking] * interface.lattice.a / np.sqrt(3), 0],
            use_cartesian_coordinates=True,
        )
        interface.name = f"Gr/hBN ({stacking}) d{distance:.2f}"
        interface.lattice.type = "HEX"
        interfaces.append(interface)
        interlayer_distance = get_average_interlayer_distance(
            interface, InterfacePartsEnum.SUBSTRATE.value, InterfacePartsEnum.FILM.value
        )
        vacuum = interface.lattice.c * (1 - max(coordinate[2] for coordinate in interface.basis.coordinates.values))
        print(f"{interface.name}: {len(interface.basis.elements.ids)} atoms, a = {interface.lattice.a:.4f} Å, "
              f"gamma = {interface.lattice.gamma:.1f}°, distance = {interlayer_distance:.3f} Å, "
              f"vacuum = {vacuum:.2f} Å, C over {' / '.join(get_registry(interface))}")

### 3.2. Preview the interfaces

In [ ]:
visualize(interfaces, repetitions=[3, 3, 1])
visualize(interfaces, repetitions=[3, 3, 1], rotation="-90x")

## 4. Pass data to the outside runtime

In [ ]:
from mat3ra.notebooks_utils.io import download_content_to_file
from mat3ra.notebooks_utils.material import set_materials

set_materials(interfaces)
for interface in interfaces:
    download_content_to_file(interface.to_json(), f"{interface.name}.json")